### Eval SRRIP rrpv value

* Maybe maxrrpv@3 ages too fast
  - Too fast to learn anything
  - Optimal should be associativy size

In [ ]:
# Data collection: Sweep vanilla SRRIP with different max RRPVs and hit deltas
python3 ./scripts/run_belady_sweep.py  \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --policies belady,lfu,srrip \
  --srrip-max-rrpvs 3,5,8,12,15 \
  --srrip-hit-deltas 1 \
  --slurm --slurm-time 04:00:00

In [8]:
# Data parsing: Collect vanilla SRRIP results
!python3 ../scripts/run_belady_sweep.py \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --policies belady,lfu,srrip \
  --srrip-max-rrpvs 3,5,8,12,15 \
  --srrip-hit-deltas 1 \
  --collect-existing-results

Total configurations: 5
Collected existing results: ok=0, missing=5
Summary written to data/belady_sweep/belady_sweep_summary.csv


In [9]:
# Plotting: Compare vanilla SRRIP variants with LFU and Belady
!python3 ../scripts/plot_belady_sweep_results.py \
  --sweep-summary-csv ./data/belady_sweep/belady_sweep_summary.csv \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --policies belady,lfu,srrip \
  --summary-csv data/belady_sweep/plot_lookup_summary.csv \
  --plot-file-format pdf

Lookup summary written to data/belady_sweep/plot_lookup_summary.csv
Found 0, missing 5
Baseline values for non-varied features:
  ew=selected_qualcomm_srv_ap|s=64|w=16
No found rows to plot.
No found rows to plot.


### Interpreting the Results

The sweep tests vanilla SRRIP with:
- **Max RRPVs**: 3, 5, 8, 12, 15
- **Hit deltas**: 1 (fast promotion toward MRU), 2 (slower promotion)

Compare the results to LFU and Belady to see if larger max-RRPV granularity helps close the gap.
If higher max RRPV values perform better, it supports your hypothesis that 4 states (max_rrpv=3) was too coarse for 16-way associativity.

***
### Try SRRIP with just a high insertion value
* This seems to be common among all good IPVs
  - Different from default srrip behavior


***
### Main Idea:
A single global vector:

IPV=[v0,v1,v2,v3,v4]

that tells SRRIP:

how aggressively to insert new cache lines
based on OPT-derived behavior

You are learning:
  - a universal cache insertion priorirty

### Finding IPV: 
* Run OPT on traces
  - You simulate ideal cache behavior
    - input: memory traces
    - output: exact eviction decisions + lifetimes

  - This gives you:
    - how long each cache line should have lived

1. Need to define reuse-distance buckets
  - Convert reuse distance → classes:

    Example:

    - D = 0–10        → very hot
    - D = 10–100      → warm
    - D = 100–1K      → cold
    - D = >1K         → dead-on-arrival

2. Observe OPT insertion outcomes
  - For every insertion track:
    - Was the block reused before eviction?
    - How long did it stay?
    - What was its reuse distance?

3. Map to IPV values
  - IPV should reflect expected usefulness at insertion
    - High reuse probability → high IPV (insert near MRU)
    - Low reuse probability → low IPV (insert near LRU)

  Example mapping:

  - Bucket            → rrpv
  - very hot          → 0
  - warm              → 1
  - cold              → 2
  - dead              → 3

* Predict probability of survival until next access
  - IPV = P(survive)

* Update on Hit
  - How to treat hits if IPV does not model state transitions.
  - RRPV_new = (1-alpha)RRPV_old + a*EIPV
  - EIPV = Sum( i * P_instr/data(i) )
  - Larger alpha = faster movement toward 0.
  - Smaller alpha = more inertia/history.



### Compare LFU, PACIPV (Best vector with exhaustive search), Belady-driven-sampling
* tuning alphas
 

In [ ]:
# Data collection for Belady's sweep. Run this before running the notebook to plot the results.
python3 ./scripts/run_belady_sweep.py \
  --train-workloads selected_qualcomm_srv_ap \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --train-fractions 1.0  \
  --policies pacipv,lfu,belady_driven_sampling   
  --pacipv-output-file-template 'data/belady_sweep/pacipv_{train_workload}_to_{eval_workload}_s{num_sets}_w{num_ways}_frac{train_fraction_tag}_alpha{bds_alpha_tag}.txt' \
  --bds-alphas 1.0,0.7,0.5,0.2,0 \
  --slurm --slurm-time 08:00:00 

SyntaxError: invalid syntax (2890822587.py, line 2)

In [ ]:
# Data parsing into the summary file.
python3 ./scripts/run_belady_sweep.py \
  --train-workloads selected_qualcomm_srv_ap \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --train-fractions 1.0 \
  --policies pacipv,lfu,belady_driven_sampling \
  --bds-alphas 1.0,0.7,0.5,0.2,0 \
  --collect-existing-results

In [ ]:
# Plotting the results after running the above data collection.
python3 ./scripts/plot_belady_sweep_results.py \
  --sweep-summary-csv ./data/belady_sweep/belady_sweep_summary.csv \
  --train-workloads selected_qualcomm_srv_ap \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --train-fractions 1.0 \
  --bds-alphas 1.0,0.7,0.5,0.2,0 \
  --policies belady_driven_sampling \
  --summary-csv data/belady_sweep/plot_lookup_summary.csv \
  --plot-file-format pdf

### Analysing Results
| Policy | Alpha | Miss Rate |
|--------|-------|-----------|
| LFU | - | 0.738919 |
| Belady-Driven Sampling | 0.0 | 0.793124 |
| Belady-Driven Sampling | 0.2 | 0.788250 |
| Belady-Driven Sampling | 0.5 | 0.788307 |
| Belady-Driven Sampling | 0.7 | 0.787458 |
| Belady-Driven Sampling | 1.0 | 0.777859 |

* Still worse than lfu and belady
* Seems best is to just choose 0 on hit (@1.0)

***
### Simulate Shadow SRRIP to find IPV (alongside belady)

We want to learn a better SRRIP policy using Belady's Optimal Algorithm as a teacher.

Instead of guessing insertion and hit behavior, we:

1. Run OPT on traces
  - This gives us perfect knowledge of future reuse
  - From it, we extract:
    - reuse distances (time until next access)
    - lifetimes (how long a line stays useful)

2. Convert OPT behavior into SRRIP states

  - We map continuous OPT behavior into discrete SRRIP states:

  - Insertion → pick the minimum RRPV that would keep a line alive long enough
  - Hit update → move a line toward the state implied by its future reuse

3. Learn two things
  - IPV (Insertion Priority Vector) → probability of inserting at each SRRIP state
  - Delta (hit correction) → how much to reduce RRPV on a hit depending on current state

4. Use these in runtime SRRIP
  - Insert using IPV
  - Update using Delta
  - Keep standard SRRIP eviction



In [ ]:
# Try to estimate the IPV with belady and use it for the srrip policy.
python ./cache_sim/estimate_belady_opt.py \
  --train-workload selected_qualcomm_srv_ap \
  --eval-workload selected_qualcomm_srv_ap \
  --num-sets 64 --num-ways 16 \
  --train-fraction 1.0 \
  --policies lfu,opt_distilled_srrip \
  --learn-opt-distilled-srrip \
  --opt-distilled-srrip-file data/belady_sweep/opt_distilled_srrip.txt \
  --slurm --slurm-time 08:00:00 

***
### Learn PACIPV via Belady + Shadow SRRIP Distillation

This policy infers PACIPV vectors from Belady next-use targets by running a shadow SRRIP state process and aggregating insertion/hit-transitions.

* On miss/insert:
  1. Find the position it should be inserted considering it's reuse distance
    - Bucket            → rrpv
    - very hot          → 0
    - warm              → 1
    - cold              → 2
    - dead              → 3
  2. Set RRPV accordingly for PTE in shadow SRRIP  
  3. If not victim found, age everyone in shadow SRRIP
    - Note: Should we try to check if Belady's choice matched SRRIP?

  - Output: vector of RRPV values with distribution
  - Choose most common for IPV[4] 

* On hit:
  1. Get current state from shadow SRRIP
    - will use it to record tansition
  2. Move to new state in the same fashion as insert
  
  - Output: A 2D array, D1: current states, D2: vector of RRPV values with distribution.
  - For each state choose the most common dist -> states transition for IPV[0,1,2,3]

* Need a way to figure out the thresholds for the buckets
  - (!) Insert a fake address with no hits every N cycles/accesses to measure how fast on average they get kicked out
    - Different profile per cache size, assoc
  
  - Analytical model to infer it from reuse distance histogram ?
    - How to model assoc? reuse distance per set?
    - Divide results in precentiles

In [ ]:
# Data collection: sweep PACIPV shadow-distilled policy (plus LFU/Belady baselines).
python3 ./scripts/run_belady_sweep.py \
  --train-workloads selected_qualcomm_srv_ap \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --train-fractions 1.0 \
  --policies belady,lfu,pacipv_shadow \
  --pacipv-shadow-max-rrpvs 3 \
  --pacipv-shadow-per-context \
  --slurm --slurm-time 08:00:00

In [ ]:
# Data parsing: collect completed PACIPV shadow sweep results into summary CSV.
python3 ./scripts/run_belady_sweep.py \
  --train-workloads selected_qualcomm_srv_ap \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --train-fractions 1.0 \
  --policies belady,lfu,pacipv_shadow \
  --pacipv-shadow-max-rrpvs 3 \
  --pacipv-shadow-per-context \
  --collect-existing-results

In [ ]:
# Plotting: compare PACIPV shadow-distilled policy against LFU and Belady.
python3 ./scripts/plot_belady_sweep_results.py \
  --sweep-summary-csv ./data/belady_sweep/belady_sweep_summary.csv \
  --train-workloads selected_qualcomm_srv_ap \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --train-fractions 1.0 \
  --policies belady,lfu,pacipv_shadow \
  --summary-csv data/belady_sweep/plot_lookup_summary.csv \
  --plot-file-format pdf

In [ ]:
# Optional sensitivity sweep: PACIPV shadow with multiple max-RRPV values.
python3 ./scripts/run_belady_sweep.py \
  --train-workloads selected_qualcomm_srv_ap \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --train-fractions 1.0 \
  --policies belady,lfu,pacipv_shadow \
  --pacipv-shadow-max-rrpvs 1,2,3 \
  --pacipv-shadow-per-context \
  --slurm --slurm-time 08:00:00

### Probabilistic IPV
* Instdead of fixed values for each IPV cell
  - Use a distributions vector [p1,p2,p3,p4] modeling how often a 